# End-to-End ML Pipeline on Snowflake
## Healthcare 30-Day Readmission Prediction — Demo Walkthrough

This notebook walks through the **complete ML lifecycle** in a single runnable session:

| Section | What | Snowflake Feature | Production Module |
|---------|------|-------------------|-------------------|
| 1 | Environment Setup | Database, Schemas, Warehouse | `src/config.py` |
| 2 | Raw Data Ingestion | Tables, `write_pandas` | notebook 02 |
| 3 | Feature Engineering | Feature Store, Dynamic Tables, Online Store | `src/feature_engineering.py` |
| 4 | Model Training | Local scikit-learn | `src/train.py` |
| 5 | Model Registration | Model Registry | `src/register_model.py` |
| 6 | Batch Inference | `mv.run()` on warehouse | `src/batch_inference.py` |
| 7 | Real-Time Inference | Online Feature Store + `mv.run()` | `src/realtime_inference.py` |
| 8 | Git Integration & Production | Git Repository, Tasks, `EXECUTE IMMEDIATE FROM` | `scripts/`, `production/` |

**Run each cell in order.** Every section is self-contained and idempotent.

---
## Section 1: Environment Setup

Create the Snowflake infrastructure: database, schemas, and warehouse.

> **Production module:** `src/config.py` — centralizes all environment settings (DEV/PROD) and session creation.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import json

# --- Connect to Snowflake ---
# In Snowflake Notebooks, the session is pre-created.
# For local use: Session.builder.config("connection_name", "DEMO").create()
session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()
print(f"Connected: {session.get_current_account()}")

In [ ]:
# --- Create infrastructure (idempotent) ---
setup_ddl = [
    "CREATE DATABASE IF NOT EXISTS HEALTHCARE_ML COMMENT = 'Healthcare ML project for 30-day readmission prediction'",
    "CREATE WAREHOUSE IF NOT EXISTS HEALTHCARE_ML_WH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE",
    "USE DATABASE HEALTHCARE_ML",
    "USE WAREHOUSE HEALTHCARE_ML_WH",
    "CREATE SCHEMA IF NOT EXISTS RAW_DATA COMMENT = 'Raw patient, admission, and clinical data'",
    "CREATE SCHEMA IF NOT EXISTS FEATURE_STORE COMMENT = 'Feature views and entities for ML'",
    "CREATE SCHEMA IF NOT EXISTS MODEL_REGISTRY COMMENT = 'Registered ML models'",
    "CREATE SCHEMA IF NOT EXISTS INFERENCE COMMENT = 'Inference results and stages'",
]

for ddl in setup_ddl:
    session.sql(ddl).collect()
    print(f"  ✓ {ddl[:60]}...")

print("\nInfrastructure ready.")

---
## Section 2: Raw Data Ingestion

Generate synthetic healthcare data and upload to Snowflake: **5,000 patients**, **10,772 admissions**, **10,772 clinical measurements**.

If the tables already exist (from a previous run), this cell will skip re-generation and just verify the data.

In [ ]:
# --- Load or generate raw data ---
session.sql("USE SCHEMA RAW_DATA").collect()

# Check if data already exists
existing = session.sql("SELECT COUNT(*) AS CNT FROM HEALTHCARE_ML.RAW_DATA.PATIENTS").collect()
if existing[0]["CNT"] > 0:
    print(f"Data already loaded: {existing[0]['CNT']} patients found. Skipping ingestion.")
    # Show row counts for verification
    for tbl in ["PATIENTS", "ADMISSIONS", "CLINICAL_MEASUREMENTS"]:
        cnt = session.sql(f"SELECT COUNT(*) AS CNT FROM HEALTHCARE_ML.RAW_DATA.{tbl}").collect()[0]["CNT"]
        print(f"  {tbl}: {cnt:,} rows")
else:
    # Generate synthetic data inline
    import random
    random.seed(42)
    np.random.seed(42)

    N_PATIENTS = 5000
    N_ADMISSIONS = 10772

    # --- Patients ---
    patients = pd.DataFrame({
        "PATIENT_ID": [f"P{str(i).zfill(5)}" for i in range(1, N_PATIENTS + 1)],
        "AGE": np.random.randint(18, 95, N_PATIENTS),
        "GENDER": np.random.choice(["M", "F"], N_PATIENTS),
        "INSURANCE_TYPE": np.random.choice(["MEDICARE", "MEDICAID", "PRIVATE", "SELF_PAY"], N_PATIENTS, p=[0.35, 0.25, 0.30, 0.10]),
        "HAS_PCP": np.random.choice([True, False], N_PATIENTS, p=[0.7, 0.3]),
    })

    # --- Admissions ---
    diagnoses = ["HEART_FAILURE", "SEPSIS", "COPD", "RENAL_FAILURE", "ACUTE_MI", "DIABETES_COMPLICATIONS", "STROKE", "PNEUMONIA", "GI_BLEED", "HIP_FRACTURE"]
    dispositions = ["HOME", "HOME_HEALTH", "REHAB", "SNF", "AMA"]
    patient_ids = np.random.choice(patients["PATIENT_ID"].values, N_ADMISSIONS)
    base_dates = pd.date_range("2023-01-01", periods=N_ADMISSIONS, freq="h")
    los = np.random.randint(1, 21, N_ADMISSIONS)

    admissions = pd.DataFrame({
        "ADMISSION_ID": [f"A{str(i).zfill(6)}" for i in range(1, N_ADMISSIONS + 1)],
        "PATIENT_ID": patient_ids,
        "ADMIT_DATE": base_dates[:N_ADMISSIONS],
        "DISCHARGE_DATE": base_dates[:N_ADMISSIONS] + pd.to_timedelta(los, unit="D"),
        "LENGTH_OF_STAY": los,
        "PRIMARY_DIAGNOSIS": np.random.choice(diagnoses, N_ADMISSIONS),
        "DISCHARGE_DISPOSITION": np.random.choice(dispositions, N_ADMISSIONS, p=[0.45, 0.20, 0.15, 0.12, 0.08]),
        "NUM_PROCEDURES": np.random.randint(0, 8, N_ADMISSIONS),
        "NUM_DIAGNOSES": np.random.randint(1, 12, N_ADMISSIONS),
        "ED_ADMISSION": np.random.choice([0, 1], N_ADMISSIONS, p=[0.6, 0.4]),
        "READMITTED_30D": np.random.choice([0, 1], N_ADMISSIONS, p=[0.75, 0.25]),
    })

    # --- Clinical measurements ---
    clinical = pd.DataFrame({
        "MEASUREMENT_ID": [f"M{str(i).zfill(6)}" for i in range(1, N_ADMISSIONS + 1)],
        "ADMISSION_ID": admissions["ADMISSION_ID"],
        "PATIENT_ID": admissions["PATIENT_ID"],
        "HEART_RATE": np.random.normal(80, 15, N_ADMISSIONS).astype(int),
        "SYSTOLIC_BP": np.random.normal(130, 20, N_ADMISSIONS).astype(int),
        "DIASTOLIC_BP": np.random.normal(80, 12, N_ADMISSIONS).astype(int),
        "TEMPERATURE": np.round(np.random.normal(98.6, 0.8, N_ADMISSIONS), 1),
        "RESPIRATORY_RATE": np.random.normal(18, 3, N_ADMISSIONS).astype(int),
        "O2_SATURATION": np.clip(np.random.normal(96, 3, N_ADMISSIONS), 80, 100).astype(int),
        "BLOOD_GLUCOSE": np.random.normal(120, 40, N_ADMISSIONS).astype(int),
        "CREATININE": np.round(np.random.normal(1.2, 0.5, N_ADMISSIONS), 2),
        "HEMOGLOBIN": np.round(np.random.normal(12.5, 2.0, N_ADMISSIONS), 1),
        "WBC_COUNT": np.round(np.random.normal(8.0, 3.0, N_ADMISSIONS), 1),
        "SODIUM": np.random.normal(140, 4, N_ADMISSIONS).astype(int),
        "POTASSIUM": np.round(np.random.normal(4.2, 0.5, N_ADMISSIONS), 1),
        "BNP": np.random.normal(300, 200, N_ADMISSIONS).astype(int),
    })

    # Upload to Snowflake
    for tbl_name, df in [("PATIENTS", patients), ("ADMISSIONS", admissions), ("CLINICAL_MEASUREMENTS", clinical)]:
        session.write_pandas(df, tbl_name, auto_create_table=True, overwrite=True)
        print(f"  Uploaded {tbl_name}: {len(df):,} rows")

    print("\nAll raw data uploaded.")

In [ ]:
# --- Quick EDA ---
print("=== Readmission Rate ===")
session.sql("""
    SELECT
        CASE WHEN READMITTED_30D = 1 THEN 'Readmitted' ELSE 'Not Readmitted' END AS STATUS,
        COUNT(*) AS COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS PCT
    FROM RAW_DATA.ADMISSIONS
    GROUP BY READMITTED_30D ORDER BY READMITTED_30D
""").show()

print("\n=== Top Diagnoses by Readmission Rate ===")
session.sql("""
    SELECT PRIMARY_DIAGNOSIS,
           COUNT(*) AS ADMISSIONS,
           ROUND(AVG(READMITTED_30D) * 100, 1) AS READMIT_PCT
    FROM RAW_DATA.ADMISSIONS
    GROUP BY PRIMARY_DIAGNOSIS
    ORDER BY READMIT_PCT DESC
""").show()

---
## Section 3: Feature Engineering & Feature Store

Create a **Feature View** backed by a Dynamic Table that joins the 3 raw tables and engineers **33 features**.
Enable the **Online Feature Store** for sub-second real-time lookups.

> **Production module:** `src/feature_engineering.py` — contains the Feature View SQL and local pandas equivalent.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity
from snowflake.ml.feature_store.feature_view import OnlineConfig

session.sql("USE SCHEMA FEATURE_STORE").collect()

# --- Initialize Feature Store ---
fs = FeatureStore(
    session=session,
    database="HEALTHCARE_ML",
    name="FEATURE_STORE",
    default_warehouse="HEALTHCARE_ML_WH",
)
print("Feature Store initialized.")

In [ ]:
# --- Register Entity ---
patient_entity = Entity(
    name="PATIENT",
    join_keys=["PATIENT_ID"],
    desc="Hospital patient identified by PATIENT_ID",
)
fs.register_entity(patient_entity)
print("Entity PATIENT registered.")
fs.list_entities().show()

In [ ]:
# --- Feature View SQL (33 engineered features) ---
FEATURE_VIEW_SQL = """
SELECT
    a.PATIENT_ID,
    a.ADMISSION_ID,
    TO_TIMESTAMP(a.DISCHARGE_DATE) AS EVENT_TIMESTAMP,
    -- Demographics (4 features)
    p.AGE,
    CASE WHEN p.GENDER = 'M' THEN 1 ELSE 0 END AS GENDER_ENC,
    CASE p.INSURANCE_TYPE
        WHEN 'MEDICAID' THEN 0 WHEN 'MEDICARE' THEN 1
        WHEN 'PRIVATE' THEN 2 WHEN 'SELF_PAY' THEN 3
    END AS INSURANCE_ENC,
    CASE WHEN p.HAS_PCP THEN 1 ELSE 0 END AS HAS_PCP_FLAG,
    -- Admission features (6)
    a.LENGTH_OF_STAY,
    a.NUM_PROCEDURES,
    a.NUM_DIAGNOSES,
    CASE a.PRIMARY_DIAGNOSIS
        WHEN 'HEART_FAILURE' THEN 3 WHEN 'SEPSIS' THEN 3
        WHEN 'COPD' THEN 2 WHEN 'RENAL_FAILURE' THEN 2
        WHEN 'ACUTE_MI' THEN 2 WHEN 'DIABETES_COMPLICATIONS' THEN 2 WHEN 'STROKE' THEN 2
        ELSE 1
    END AS DIAGNOSIS_RISK_SCORE,
    CASE a.DISCHARGE_DISPOSITION
        WHEN 'HOME' THEN 0 WHEN 'HOME_HEALTH' THEN 1 WHEN 'REHAB' THEN 1
        WHEN 'SNF' THEN 2 WHEN 'AMA' THEN 3
    END AS DISPOSITION_RISK_SCORE,
    a.ED_ADMISSION,
    -- Clinical vitals & labs (13)
    c.HEART_RATE, c.SYSTOLIC_BP, c.DIASTOLIC_BP, c.TEMPERATURE,
    c.RESPIRATORY_RATE, c.O2_SATURATION, c.BLOOD_GLUCOSE, c.CREATININE,
    c.HEMOGLOBIN, c.WBC_COUNT, c.SODIUM, c.POTASSIUM, c.BNP,
    -- Abnormality flags (7)
    CASE WHEN c.HEART_RATE > 100 OR c.HEART_RATE < 60 THEN 1 ELSE 0 END AS ABNORMAL_HR,
    CASE WHEN c.SYSTOLIC_BP > 160 OR c.SYSTOLIC_BP < 90 THEN 1 ELSE 0 END AS ABNORMAL_BP,
    CASE WHEN c.O2_SATURATION < 93 THEN 1 ELSE 0 END AS LOW_O2,
    CASE WHEN c.CREATININE > 1.5 THEN 1 ELSE 0 END AS HIGH_CREATININE,
    CASE WHEN c.HEMOGLOBIN < 10 THEN 1 ELSE 0 END AS LOW_HEMOGLOBIN,
    CASE WHEN c.BNP > 500 THEN 1 ELSE 0 END AS HIGH_BNP,
    CASE WHEN c.BLOOD_GLUCOSE > 200 OR c.BLOOD_GLUCOSE < 70 THEN 1 ELSE 0 END AS ABNORMAL_GLUCOSE,
    -- Historical features (3) via window functions
    COALESCE(COUNT(*) OVER (
        PARTITION BY a.PATIENT_ID ORDER BY a.ADMIT_DATE
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ), 0) AS PRIOR_ADMISSIONS_6M,
    COALESCE(SUM(a.READMITTED_30D) OVER (
        PARTITION BY a.PATIENT_ID ORDER BY a.ADMIT_DATE
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ), 0) AS PRIOR_READMISSIONS,
    COALESCE(AVG(a.LENGTH_OF_STAY) OVER (
        PARTITION BY a.PATIENT_ID ORDER BY a.ADMIT_DATE
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ), 0) AS AVG_PRIOR_LOS,
    -- Target
    a.READMITTED_30D
FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS a
JOIN HEALTHCARE_ML.RAW_DATA.PATIENTS p ON a.PATIENT_ID = p.PATIENT_ID
JOIN HEALTHCARE_ML.RAW_DATA.CLINICAL_MEASUREMENTS c
    ON a.ADMISSION_ID = c.ADMISSION_ID AND a.PATIENT_ID = c.PATIENT_ID
"""

print("Feature View SQL defined (33 features + target).")
print(f"SQL length: {len(FEATURE_VIEW_SQL)} chars")

In [ ]:
# --- Create Feature View backed by Dynamic Table ---
feature_df = session.sql(FEATURE_VIEW_SQL)

fv = FeatureView(
    name="PATIENT_CLINICAL_FEATURES",
    entities=[patient_entity],
    feature_df=feature_df,
    timestamp_col="EVENT_TIMESTAMP",
    refresh_freq="5 minutes",
    desc="Patient demographics, admission details, clinical vitals, and historical features for readmission prediction",
)

fv = fs.register_feature_view(
    feature_view=fv,
    version="V1",
    overwrite=True,
)
print(f"Feature View registered: {fv.name} V{fv.version}")
print(f"Backing Dynamic Table: PATIENT_CLINICAL_FEATURES$V1")
print(f"Refresh frequency: 5 minutes")

In [ ]:
# --- Enable Online Feature Store ---
fv = fs.get_feature_view("PATIENT_CLINICAL_FEATURES", "V1")
fv = fs.update_feature_view(
    name="PATIENT_CLINICAL_FEATURES",
    version="V1",
    online_config=OnlineConfig(enable=True, target_lag="1 minute"),
)
print("Online Feature Store enabled (1-minute target lag).")
print("This provides sub-second key-value lookups for real-time inference.")

In [ ]:
# --- Preview features ---
print("=== Feature Store Preview (5 rows) ===")
session.table('HEALTHCARE_ML.FEATURE_STORE."PATIENT_CLINICAL_FEATURES$V1"').show(5)

row_count = session.table('HEALTHCARE_ML.FEATURE_STORE."PATIENT_CLINICAL_FEATURES$V1"').count()
print(f"\nTotal rows in Feature Store: {row_count:,}")

---
## Section 4: Model Training

Train a **GradientBoostingClassifier** (scikit-learn) on the feature data.
We pull training data from Snowflake, train locally, and evaluate.

> **Production module:** `src/train.py` — training logic extracted for reuse and scheduled retraining.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report, confusion_matrix
)
import joblib

# --- Feature columns (single source of truth) ---
FEATURE_COLUMNS = [
    "AGE", "GENDER_ENC", "INSURANCE_ENC", "HAS_PCP_FLAG",
    "LENGTH_OF_STAY", "NUM_PROCEDURES", "NUM_DIAGNOSES",
    "DIAGNOSIS_RISK_SCORE", "DISPOSITION_RISK_SCORE", "ED_ADMISSION",
    "HEART_RATE", "SYSTOLIC_BP", "DIASTOLIC_BP", "TEMPERATURE",
    "RESPIRATORY_RATE", "O2_SATURATION", "BLOOD_GLUCOSE", "CREATININE",
    "HEMOGLOBIN", "WBC_COUNT", "SODIUM", "POTASSIUM", "BNP",
    "ABNORMAL_HR", "ABNORMAL_BP", "LOW_O2", "HIGH_CREATININE",
    "LOW_HEMOGLOBIN", "HIGH_BNP", "ABNORMAL_GLUCOSE",
    "PRIOR_ADMISSIONS_6M", "PRIOR_READMISSIONS", "AVG_PRIOR_LOS",
]
TARGET = "READMITTED_30D"

print(f"Features: {len(FEATURE_COLUMNS)}")
print(f"Target: {TARGET}")

In [ ]:
# --- Pull training data from Snowflake ---
train_df = session.table('HEALTHCARE_ML.FEATURE_STORE."PATIENT_CLINICAL_FEATURES$V1"').to_pandas()
print(f"Pulled {len(train_df):,} rows from Feature Store")
print(f"Readmission rate: {train_df[TARGET].mean():.1%}")

X = train_df[FEATURE_COLUMNS].astype(float)
y = train_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

In [ ]:
# --- Train GradientBoostingClassifier ---
model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42,
)

print("Training GradientBoostingClassifier (200 estimators, max_depth=4)...")
model.fit(X_train, y_train)
print("Training complete.")

# --- Evaluate ---
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

print(f"\n=== Model Evaluation ===")
print(f"ROC AUC:           {roc_auc:.4f}")
print(f"Average Precision: {avg_precision:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Not Readmitted', 'Readmitted'])}")

In [ ]:
# --- Model artifacts (in-memory for Snowflake Notebooks) ---
# In production, artifacts are saved locally via src/train.py
# Here we keep everything in memory for the demo.

# Feature importance (top 10)
importance = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS)
print("=== Top 10 Features ===")
print(importance.sort_values(ascending=False).head(10).to_string())

# Metadata (kept in memory for model registration)
metadata = {
    "model_type": "GradientBoostingClassifier",
    "features": FEATURE_COLUMNS,
    "target": TARGET,
    "model_metrics": {"roc_auc": round(roc_auc, 4), "average_precision": round(avg_precision, 4)},
    "training_samples": len(X_train),
    "test_samples": len(X_test),
}
print(f"\nModel metrics: ROC AUC={roc_auc:.4f}, Avg Precision={avg_precision:.4f}")
print(f"Training: {len(X_train):,} | Test: {len(X_test):,}")

---
## Section 5: Model Registration

Register the trained model in the **Snowflake Model Registry**.
This auto-detects inference functions: `PREDICT`, `PREDICT_PROBA`, `EXPLAIN`, etc.

> **Production module:** `src/register_model.py` — loads joblib and registers via `registry.log_model()`.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import task as ml_task

session.sql("USE SCHEMA MODEL_REGISTRY").collect()

# --- Initialize Registry ---
registry = Registry(
    session=session,
    database_name="HEALTHCARE_ML",
    schema_name="MODEL_REGISTRY",
)
print("Registry initialized.")

# --- Prepare sample input for schema inference ---
sample_input = X_test.head(100)
print(f"Sample input: {sample_input.shape}")

In [ ]:
# --- Register model ---
MODEL_NAME = "READMISSION_PREDICTOR"
VERSION = "V1"

# Check if this version already exists
try:
    mv = registry.get_model(MODEL_NAME).version(VERSION)
    print(f"Model {MODEL_NAME} {VERSION} already exists. Skipping registration.")
except Exception:
    print(f"Registering {MODEL_NAME} {VERSION}...")
    mv = registry.log_model(
        model=model,
        model_name=MODEL_NAME,
        version_name=VERSION,
        sample_input_data=sample_input,
        conda_dependencies=["scikit-learn"],
        comment=f"30-day readmission predictor. GradientBoostingClassifier, {len(FEATURE_COLUMNS)} features. ROC AUC={roc_auc:.4f}",
        metrics={
            "roc_auc": round(roc_auc, 4),
            "average_precision": round(avg_precision, 4),
            "training_samples": len(X_train),
            "test_samples": len(X_test),
            "n_features": len(FEATURE_COLUMNS),
        },
        task=ml_task.Task.TABULAR_BINARY_CLASSIFICATION,
    )
    print(f"Registered: {MODEL_NAME} {VERSION}")

print(f"Functions: {[f.name for f in mv.show_functions()]}")

In [ ]:
# --- Verify: run predict on 5 rows ---
test_preds = mv.run(X_test.head(5), function_name="predict")
print("=== Verification: predict on 5 rows ===")
print(test_preds)

test_proba = mv.run(X_test.head(5), function_name="predict_proba")
print("\n=== Verification: predict_proba on 5 rows ===")
print(test_proba)

---
## Section 6: Batch Inference

Score **all patients** using the registered model with warehouse compute (`mv.run()`).
Results are saved to `HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS`.

> **Production module:** `src/batch_inference.py` — supports both `mv.run()` (warehouse) and `mv.run_batch()` (SPCS/Ray) for larger datasets.

For this demo we use warehouse compute. For millions of rows, switch to:
```python
mv.run_batch(compute_pool="DEMO_POOL_CPU", X=features, ...)
```

In [ ]:
session.sql("USE SCHEMA INFERENCE").collect()

# --- Load model from registry ---
mv = registry.get_model(MODEL_NAME).version(VERSION)
print(f"Model loaded: {MODEL_NAME} {VERSION}")

# --- Read features from Dynamic Table ---
feature_table = session.table('HEALTHCARE_ML.FEATURE_STORE."PATIENT_CLINICAL_FEATURES$V1"')
features_only = feature_table.select(FEATURE_COLUMNS)
row_count = features_only.count()
print(f"Input rows: {row_count:,}")

In [ ]:
# --- Run batch inference on warehouse ---
import time
start = time.time()

print("Running batch inference via mv.run() on warehouse...")
predictions = mv.run(features_only, function_name="predict_proba")
elapsed = time.time() - start
print(f"Inference complete in {elapsed:.1f}s")

# Identify columns — predict_proba always returns 2 columns:
# [P(class_0), P(class_1)] where class_1 = readmitted
pred_cols = predictions.columns
# The readmission probability is always the LAST column
READMIT_PROB_COL = pred_cols[-1]
print(f"Prediction columns: {pred_cols}")
print(f"Readmission probability column: {READMIT_PROB_COL}")

# --- Save results ---
predictions.write.save_as_table(
    "HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS",
    mode="overwrite",
)
count = session.sql("SELECT COUNT(*) AS CNT FROM HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS").collect()[0]["CNT"]
print(f"Saved {count:,} predictions to BATCH_PREDICTIONS")

In [ ]:
# --- Analyze results ---
# Get actual column names from the predictions table
pred_table_cols = [c.name for c in session.table("HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS").schema.fields]
print(f"Prediction table columns: {pred_table_cols}")

# predict_proba output is always [P(not_readmitted), P(readmitted)]
# The readmission probability is the LAST column
prob_col = pred_table_cols[-1]
print(f"Using column: {prob_col}")

print("\n=== Prediction Summary ===")
session.sql(f"""
    SELECT
        COUNT(*) AS TOTAL_SCORED,
        ROUND(AVG(\"{prob_col}\"), 4) AS AVG_READMISSION_PROB,
        ROUND(MIN(\"{prob_col}\"), 4) AS MIN_PROB,
        ROUND(MAX(\"{prob_col}\"), 4) AS MAX_PROB,
        SUM(CASE WHEN \"{prob_col}\" > 0.5 THEN 1 ELSE 0 END) AS HIGH_RISK_COUNT
    FROM HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS
""").show()

print("\n=== Risk Tier Breakdown ===")
session.sql(f"""
    SELECT
        CASE
            WHEN \"{prob_col}\" >= 0.7 THEN 'HIGH (>=0.7)'
            WHEN \"{prob_col}\" >= 0.4 THEN 'MEDIUM (0.4-0.7)'
            ELSE 'LOW (<0.4)'
        END AS RISK_TIER,
        COUNT(*) AS PATIENT_COUNT,
        ROUND(AVG(\"{prob_col}\"), 4) AS AVG_PROBABILITY
    FROM HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS
    GROUP BY RISK_TIER
    ORDER BY AVG_PROBABILITY DESC
""").show()

---
## Section 7: Real-Time Inference

For single-patient lookups, the **Online Feature Store** retrieves features in sub-second,
and `mv.run()` scores immediately. This is the pattern for point-of-care risk assessment.

> **Production module:** `src/realtime_inference.py` — wraps the feature lookup + scoring into a single function.

In [ ]:
# --- Pick sample patients ---
sample_patients = session.sql("""
    SELECT DISTINCT PATIENT_ID
    FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
    ORDER BY PATIENT_ID
    LIMIT 5
""").to_pandas()["PATIENT_ID"].tolist()

print(f"Sample patients: {sample_patients}")

In [ ]:
# --- Real-time feature retrieval + inference ---
fv = fs.get_feature_view("PATIENT_CLINICAL_FEATURES", "V1")
mv = registry.get_model(MODEL_NAME).version(VERSION)

print("=== Real-Time Inference (per patient) ===")
print(f"{'PATIENT_ID':<15} {'RISK_PROB':>10} {'RISK_LEVEL':>12} {'LATENCY':>10}")
print("-" * 50)

for patient_id in sample_patients:
    start = time.time()

    # 1. Retrieve features from Online Feature Store
    spine = session.create_dataframe([{"PATIENT_ID": patient_id}])
    features = fs.retrieve_feature_values(
        spine_df=spine, features=[fv], spine_timestamp_col=None
    )

    # 2. Score with registered model
    feature_input = features.select(FEATURE_COLUMNS)
    pred = mv.run(feature_input, function_name="predict_proba").to_pandas()

    elapsed = time.time() - start

    # 3. predict_proba returns [P(class_0), P(class_1)]
    #    Last column is always the readmission probability
    prob = float(pred.iloc[0, -1])
    risk = "HIGH" if prob >= 0.5 else "MEDIUM" if prob >= 0.3 else "LOW"

    print(f"{patient_id:<15} {prob:>10.4f} {risk:>12} {elapsed:>8.2f}s")

---
## Section 8: Git Integration & Production Scheduling

The code above is what runs during development. For **production**, it's extracted into
`src/` modules and executed directly from Git inside Snowflake.

### The Promotion Flow
```
notebooks/  -->  src/  -->  production/  -->  Snowflake Tasks
(experiment)    (modules)    (entry points)    (scheduled)
```

### How Snowflake executes code from Git
1. A **Git Repository** object connects to your GitHub repo
2. `ALTER GIT REPOSITORY ... FETCH` pulls the latest code
3. `EXECUTE IMMEDIATE FROM @repo/branches/main/production/run_batch_inference.py` runs it
4. **Snowflake Tasks** automate the fetch + execute on a schedule

In [ ]:
# --- Git Integration Setup (reference SQL) ---
# These commands were already run to set up the integration.
# Shown here for documentation.

git_setup_sql = """
-- 1. API Integration (account-level, one-time)
CREATE OR REPLACE API INTEGRATION GITHUB_API_INTEGRATION
    API_PROVIDER = git_https_api
    API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-moahmed/')
    ALLOWED_AUTHENTICATION_SECRETS = ALL
    ENABLED = TRUE;

-- 2. Git Repository object
CREATE OR REPLACE GIT REPOSITORY
    HEALTHCARE_ML.GIT_INTEGRATION.HEALTHCARE_ML_REPO
    ORIGIN = 'https://github.com/sfc-gh-moahmed/healthcare-readmission-ml.git'
    API_INTEGRATION = GITHUB_API_INTEGRATION;

-- 3. Fetch latest code
ALTER GIT REPOSITORY HEALTHCARE_ML.GIT_INTEGRATION.HEALTHCARE_ML_REPO FETCH;

-- 4. Execute production scripts from Git
EXECUTE IMMEDIATE FROM
    @HEALTHCARE_ML.GIT_INTEGRATION.HEALTHCARE_ML_REPO
    /branches/main/production/run_batch_inference.py;
"""
print(git_setup_sql)

In [ ]:
# --- Task Scheduling (reference SQL) ---
task_sql = """
-- Task 1: Keep code in sync (every 60 min)
CREATE OR REPLACE TASK HEALTHCARE_ML.TASKS.GIT_FETCH_TASK
    WAREHOUSE = HEALTHCARE_ML_WH
    SCHEDULE  = '60 MINUTE'
AS
    ALTER GIT REPOSITORY
        HEALTHCARE_ML.GIT_INTEGRATION.HEALTHCARE_ML_REPO FETCH;

-- Task 2: Score patients (chained after fetch)
CREATE OR REPLACE TASK HEALTHCARE_ML.TASKS.BATCH_SCORING_TASK
    WAREHOUSE = HEALTHCARE_ML_WH
    AFTER HEALTHCARE_ML.TASKS.GIT_FETCH_TASK
AS
    EXECUTE IMMEDIATE FROM
        @HEALTHCARE_ML.GIT_INTEGRATION.HEALTHCARE_ML_REPO
        /branches/main/production/run_batch_inference.py;

-- Enable tasks (child first, then root)
ALTER TASK BATCH_SCORING_TASK RESUME;
ALTER TASK GIT_FETCH_TASK RESUME;
"""
print(task_sql)

In [ ]:
# --- Verify everything is in place ---
print("="*60)
print("  DEMO WALKTHROUGH COMPLETE")
print("="*60)

checks = [
    ("Database",       "SHOW DATABASES LIKE 'HEALTHCARE_ML'"),
    ("Raw Tables",     "SELECT COUNT(*) AS CNT FROM HEALTHCARE_ML.RAW_DATA.PATIENTS"),
    ("Feature Store",  "SHOW DYNAMIC TABLES LIKE '%PATIENT_CLINICAL%' IN SCHEMA HEALTHCARE_ML.FEATURE_STORE"),
    ("Model Registry", "SHOW MODELS IN HEALTHCARE_ML.MODEL_REGISTRY"),
    ("Predictions",    "SELECT COUNT(*) AS CNT FROM HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS"),
]

for name, sql in checks:
    result = session.sql(sql).collect()
    status = "PASS" if len(result) > 0 else "FAIL"
    print(f"  [{status}] {name}")

print("\n  GitHub: https://github.com/sfc-gh-moahmed/healthcare-readmission-ml")
print("  Streamlit: HEALTHCARE_ML.PUBLIC.HEALTHCARE_ML_PIPELINE_DEMO")
print("="*60)

In [ ]:
# --- Done! ---
# In Snowflake Notebooks, do NOT close the session (it's managed by the platform).
# For local use: session.close()
print("Demo walkthrough complete. All pipeline stages executed successfully.")